# Проверки `c_nazn` в `ods.scd1_z_main_docum`

Цели:
1. Посмотреть все уникальные варианты `c_nazn` за первую неделю июня.
2. Выделить варианты `c_nazn`, которые могут относиться к эквайрингу.

Период задается параметрами ниже (по умолчанию: с `2026-06-01` по `2026-06-07` включительно).

In [ ]:
import re
import time

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect


def normalize_text_series(text_series):
    normalized = text_series.fillna('').astype(str)
    normalized = normalized.str.replace('\u00A0', ' ', regex=False)
    normalized = normalized.str.replace('\u202F', ' ', regex=False)
    normalized = normalized.str.replace('\u2007', ' ', regex=False)
    normalized = normalized.str.lower().str.replace('ё', 'е', regex=False)
    normalized = normalized.str.replace(r'\s+', ' ', regex=True).str.strip()
    return normalized


def cut_tail_for_semantics_series(text_series, contract_tail_cut_regex=r'(?i)(\bпо\s+договору\b).*$'):
    # Regex применяем только к строкам, где точно встречается "по договору".
    contract_mask = text_series.str.contains('по договору', regex=False, na=False)
    if not contract_mask.any():
        return text_series, contract_mask

    semantic = text_series.copy()
    semantic.loc[contract_mask] = semantic.loc[contract_mask].str.replace(
        contract_tail_cut_regex, r'\1', regex=True
    )
    semantic.loc[contract_mask] = semantic.loc[contract_mask].str.strip(' ,;:-').str.replace(
        r'\s+', ' ', regex=True
    )
    return semantic, contract_mask


def make_semantic_key_series(text_series):
    return text_series.str.replace(r'[^0-9a-zа-я]+', '', regex=True)


print('Imports and optimized text helpers loaded')

In [ ]:
# Параметры периода: первая неделя июня
week_start = '2026-06-01'
week_end_exclusive = '2026-06-08'  # c 01 по 07 июня включительно

# Таблица-источник
table_name = 'ods.scd1_z_main_docum'

# Параметры подключения к Impala (как в 01_07_acq_dash.ipynb)
impala_db = 'sandbox_ai'
impala_queue = 'ai'
impala_user_name = 'Shestopalov-VYur'
impala_keytab_path = '/home/jovyan/test_requests/tech.keytab'
impala_use_credentials = True
impala_update_keytab = True

# Параметры выполнения
mem_limit = '8g'

# Правила семантической унификации
apply_contract_tail_cut = True
contract_tail_cut_regex = r'(?i)(\bпо\s+договору\b).*$'
use_semantic_key = True  # ключ без пробелов/знаков препинания
compute_expensive_stats = False  # False -> без тяжелых nunique по всем строкам

# Ловим слова от корня "эквайр" (эквайринг, эквайринга, эквайринговый и т.д.)
ekv_pattern_py = r'(?:^|[^а-яa-z0-9])эквайр[а-я]*(?:[^а-яa-z0-9]|$)'
use_fast_ekv_prefilter = True  # сначала contains('эквайр'), regex только по подмножеству

# Параметры отображения таблиц
show_full_tables = False
preview_limit = 100
show_all_ekv_raw_variants = True  # до общей обработки: показать все c_nazn с %эквайр%

if show_full_tables:
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
else:
    pd.set_option('display.max_rows', preview_limit)
    pd.set_option('display.max_columns', 50)
    pd.set_option('display.max_colwidth', 140)

pd.set_option('display.width', 0)

print(f'Period: [{week_start}, {week_end_exclusive})')
print(f'table={table_name}')
print(
    f'show_full_tables={show_full_tables}, preview_limit={preview_limit}, '
    f'apply_contract_tail_cut={apply_contract_tail_cut}, use_semantic_key={use_semantic_key}, '
    f'use_fast_ekv_prefilter={use_fast_ekv_prefilter}, compute_expensive_stats={compute_expensive_stats}, '
    f'show_all_ekv_raw_variants={show_all_ekv_raw_variants}'
)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': impala_queue},
    kerberos={
        'keytab_path': impala_keytab_path,
        'use_credentials': impala_use_credentials,
        'update_keytab': impala_update_keytab,
    },
    user_params={'user_name': impala_user_name},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Один SQL в Impala: выгружаем все варианты `c_nazn`

In [ ]:
sql_base_variants = f"""
select
    coalesce(c_nazn, '') as c_nazn_raw,
    count(*) as raw_cnt
from {table_name}
where cast(c_date_prov as date) >= date '{week_start}'
  and cast(c_date_prov as date) < date '{week_end_exclusive}'
group by coalesce(c_nazn, '')
"""

fetch_started_at = time.perf_counter()
with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    base_variants_df = imp.fetch(sql_base_variants)
fetch_elapsed_sec = round(time.perf_counter() - fetch_started_at, 2)

if 'c_nazn_raw' not in base_variants_df.columns or 'raw_cnt' not in base_variants_df.columns:
    raise RuntimeError('base_variants_df must contain columns: c_nazn_raw, raw_cnt')

base_variants_df['c_nazn_raw'] = base_variants_df['c_nazn_raw'].fillna('').astype(str)
base_variants_df['raw_cnt'] = pd.to_numeric(base_variants_df['raw_cnt'], errors='coerce').fillna(0).astype('int64')

print(f'Impala fetch done in: {fetch_elapsed_sec}s')
print(f'Rows in base_variants_df: {len(base_variants_df):,}')
print(f'Total scoped rows by count: {int(base_variants_df["raw_cnt"].sum()):,}')

if show_full_tables:
    base_variants_preview_df = base_variants_df.sort_values(
        ['raw_cnt', 'c_nazn_raw'], ascending=[False, True], kind='stable'
    )
else:
    base_variants_preview_df = base_variants_df.nlargest(preview_limit, 'raw_cnt')
    base_variants_preview_df = base_variants_preview_df.sort_values(
        ['raw_cnt', 'c_nazn_raw'], ascending=[False, True], kind='stable'
    )

display(base_variants_preview_df)

## 2) Проверка до общей обработки: все уникальные `c_nazn` с `%эквайр%`

In [ ]:
if 'base_variants_df' not in globals():
    raise RuntimeError('Run base variants load cell first')

ekv_raw_check_started_at = time.perf_counter()

# Проверка до общей обработки: SQL-like фильтр %эквайр% по уже выгруженной базе вариантов.
ekv_raw_mask = base_variants_df['c_nazn_raw'].str.contains('эквайр', case=False, regex=False, na=False)
ekv_raw_variants_df = base_variants_df.loc[ekv_raw_mask, ['c_nazn_raw', 'raw_cnt']].copy()
ekv_raw_variants_df = ekv_raw_variants_df.rename(columns={'c_nazn_raw': 'c_nazn', 'raw_cnt': 'cnt'})
ekv_raw_variants_df = ekv_raw_variants_df.sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable')

print(f'ekv raw check done in: {round(time.perf_counter() - ekv_raw_check_started_at, 2)}s')
print(f'Rows in ekv_raw_variants_df: {len(ekv_raw_variants_df):,}')
print(f'Total rows by count for %эквайр%: {int(ekv_raw_variants_df["cnt"].sum()):,}')

if show_all_ekv_raw_variants or show_full_tables:
    display(ekv_raw_variants_df)
else:
    display(ekv_raw_variants_df.head(preview_limit))

In [ ]:
if 'base_variants_df' not in globals():
    raise RuntimeError('Run base variants load cell first')

python_stage_started_at = time.perf_counter()

# Линейная обработка только в pandas (без полной сортировки 5.9M строк)
work_variants_df = base_variants_df[['c_nazn_raw', 'raw_cnt']].copy()

norm_started_at = time.perf_counter()
work_variants_df['c_nazn_norm'] = normalize_text_series(work_variants_df['c_nazn_raw'])
print(f'normalize_text_series: {round(time.perf_counter() - norm_started_at, 2)}s')

if apply_contract_tail_cut:
    cut_started_at = time.perf_counter()
    semantic_series, contract_mask = cut_tail_for_semantics_series(
        work_variants_df['c_nazn_norm'],
        contract_tail_cut_regex,
    )
    work_variants_df['c_nazn_semantic'] = semantic_series
    print(
        f'contract tail cut: {round(time.perf_counter() - cut_started_at, 2)}s, '
        f'rows_with_phrase={int(contract_mask.sum()):,}'
    )
else:
    work_variants_df['c_nazn_semantic'] = work_variants_df['c_nazn_norm']

if use_semantic_key:
    key_started_at = time.perf_counter()
    non_alnum_mask = work_variants_df['c_nazn_semantic'].str.contains(r'[^0-9a-zа-я]', regex=True, na=False)
    key_series = work_variants_df['c_nazn_semantic'].copy()
    key_series.loc[non_alnum_mask] = make_semantic_key_series(key_series.loc[non_alnum_mask])
    work_variants_df['c_nazn_key'] = key_series
    print(
        f'make_semantic_key_series: {round(time.perf_counter() - key_started_at, 2)}s, '
        f'rows_with_non_alnum={int(non_alnum_mask.sum()):,}'
    )
else:
    work_variants_df['c_nazn_key'] = work_variants_df['c_nazn_semantic']

# Быстрый prefilter: regex по ekv_pattern_py только для строк, где есть "эквайр".
ekv_started_at = time.perf_counter()
if use_fast_ekv_prefilter:
    fast_mask = work_variants_df['c_nazn_norm'].str.contains('эквайр', regex=False, na=False)
    is_ekv = pd.Series(False, index=work_variants_df.index)
    is_ekv.loc[fast_mask] = work_variants_df.loc[fast_mask, 'c_nazn_norm'].str.contains(
        ekv_pattern_py,
        regex=True,
        na=False,
    )
    work_variants_df['is_ekv'] = is_ekv
else:
    work_variants_df['is_ekv'] = work_variants_df['c_nazn_norm'].str.contains(
        ekv_pattern_py,
        regex=True,
        na=False,
    )
print(f'ekv mask build: {round(time.perf_counter() - ekv_started_at, 2)}s')

agg_started_at = time.perf_counter()
unique_variants_df = (
    work_variants_df.groupby('c_nazn_key', dropna=False, as_index=False, sort=False)
    .agg(
        c_nazn=('c_nazn_semantic', 'first'),
        cnt=('raw_cnt', 'sum'),
        raw_variants_collapsed=('c_nazn_raw', 'nunique'),
    )
)
print(f'unique groupby: {round(time.perf_counter() - agg_started_at, 2)}s')

if compute_expensive_stats:
    unique_semantic_count = int(work_variants_df['c_nazn_semantic'].nunique(dropna=False))
else:
    unique_semantic_count = -1

unique_stats_df = pd.DataFrame([
    {
        'total_rows': int(work_variants_df['raw_cnt'].sum()),
        'unique_raw_c_nazn_count': int(len(work_variants_df)),
        'unique_semantic_count': unique_semantic_count,
        'unique_key_count': int(len(unique_variants_df)),
    }
])

print(f'python stage total: {round(time.perf_counter() - python_stage_started_at, 2)}s')

display(unique_stats_df)
print(f'Rows in unique_variants_df: {len(unique_variants_df):,}')
if not compute_expensive_stats:
    print('unique_semantic_count skipped (compute_expensive_stats=False)')

if show_full_tables:
    display(unique_variants_df.sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable'))
else:
    unique_preview_df = unique_variants_df.nlargest(preview_limit, 'cnt')
    unique_preview_df = unique_preview_df.sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable')
    display(unique_preview_df)

## 3) Варианты `c_nazn` по эквайрингу (из обработанного `work_variants_df`)

In [ ]:
if 'work_variants_df' not in globals():
    raise RuntimeError('Run unique variants cell first')

ekv_scope_df = work_variants_df[work_variants_df['is_ekv']]

if compute_expensive_stats:
    matched_unique_semantic = int(ekv_scope_df['c_nazn_semantic'].nunique(dropna=False))
else:
    matched_unique_semantic = -1

ekv_stats_df = pd.DataFrame([
    {
        'scoped_rows': int(work_variants_df['raw_cnt'].sum()),
        'matched_rows': int(ekv_scope_df['raw_cnt'].sum()),
        'matched_unique_semantic': matched_unique_semantic,
        'matched_unique_key': int(ekv_scope_df['c_nazn_key'].nunique(dropna=False)),
    }
])

print(f'Rows in ekv_scope_df: {len(ekv_scope_df):,}')
if not compute_expensive_stats:
    print('matched_unique_semantic skipped (compute_expensive_stats=False)')
display(ekv_stats_df)

In [ ]:
if 'work_variants_df' not in globals():
    raise RuntimeError('Run unique variants cell first')

ekv_scope_df = work_variants_df[work_variants_df['is_ekv']]

ekv_group_started_at = time.perf_counter()
ekv_variants_df = (
    ekv_scope_df.groupby('c_nazn_key', dropna=False, as_index=False, sort=False)
    .agg(
        c_nazn=('c_nazn_semantic', 'first'),
        cnt=('raw_cnt', 'sum'),
        raw_variants_collapsed=('c_nazn_raw', 'nunique'),
    )
)
print(f'ekv groupby: {round(time.perf_counter() - ekv_group_started_at, 2)}s')
print(f'Rows in ekv_variants_df: {len(ekv_variants_df):,}')

if show_full_tables:
    display(ekv_variants_df.sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable'))
else:
    ekv_preview_df = ekv_variants_df.nlargest(preview_limit, 'cnt')
    ekv_preview_df = ekv_preview_df.sort_values(['cnt', 'c_nazn'], ascending=[False, True], kind='stable')
    display(ekv_preview_df)

In [ ]:
# Необязательно: сохранить результаты в CSV
save_to_csv = False
unique_out_path = './c_nazn_unique_first_week_june.csv'
ekv_out_path = './c_nazn_ekv_first_week_june.csv'

if save_to_csv:
    unique_variants_df.to_csv(unique_out_path, index=False)
    ekv_variants_df.to_csv(ekv_out_path, index=False)
    print(f'Saved: {unique_out_path}')
    print(f'Saved: {ekv_out_path}')
else:
    print('save_to_csv=False, nothing was written.')